# 03 · Monte Carlo of simulated landings (Colab or Kaggle, GPU)

Each landing is one closed-loop flight: Blender camera frames, NFT navigation, Matchpoint-style burn, back-away decision. The same flight is judged four ways (prior map complete or stale, with or without the neural hazard detector).

**Sharding:** set `SEEDS` below per session (for example 0-250, 250-500, ...). Results are one JSON file per landing in Google Drive; re-running a cell skips finished landings, so a disconnect loses at most one flight.

Needs the trained network from notebook 02 (`best.pt`). Leave `CKPT = None` to run the baseline only.

In [ ]:
SEEDS = (0, 250)          # change per session
CKPT = '/content/drive/MyDrive/touchdown/runs/unet_v1/best.pt'   # or None
OUT = '/content/drive/MyDrive/touchdown/montecarlo/v1'

!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libxkbcommon0 libsm6 > /dev/null
!wget -q https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz -O /content/blender.tar.xz
!mkdir -p /content/blender && tar -xf /content/blender.tar.xz -C /content/blender --strip-components=1
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /content/touch_down || (cd /content/touch_down && git pull -q)
%cd /content/touch_down
!pip install -q -e .
import os
os.environ['TOUCHDOWN_DATA'] = '/content/data'
os.environ['TOUCHDOWN_BLENDER'] = '/content/blender/blender'
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p {OUT}

In [ ]:
# NASA terrain model -> height grid (about 3 minutes)
!python scripts/fetch_data.py
!python scripts/build_dtm.py

In [ ]:
ck = f'--checkpoint {CKPT}' if CKPT else ''
!python -m touchdown.sim.montecarlo --out {OUT} --seeds {SEEDS[0]} {SEEDS[1]} --device GPU {ck}

In [ ]:
# Quick look at what has finished
import glob, json, numpy as np
R = [json.load(open(f)) for f in sorted(glob.glob(f'{OUT}/landing_*.json'))]
d = np.array([r['delivery_error_m'] for r in R])
print(len(R), 'landings | delivery error median', np.median(d).round(2), 'm, 95th pct', np.percentile(d, 95).round(2), 'm')
print('unsafe contact if we never aborted:', np.mean([r['unsafe_contact'] for r in R]).round(3))